# Community Detection Example (Mall Customers Dataset)

**Goal: Find communities of similar customers by treating the dataset as a similarity network.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/unsupervised/ at the repo root
SRC_UNSUP = os.path.join(REPO_ROOT, 'src', 'unsupervised')
sys.path.insert(0, SRC_UNSUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
from community_detection import LabelPropagation
from pca import PCA
from k_means_clustering import KMeans
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv(os.path.join(DATA_DIR, 'Mall_Customers.csv'))
MALL_FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X_raw = mall[MALL_FEATURES].values.astype(float)
X = StandardScaler().fit_transform(X_raw)
print(f"Dataset loaded: {mall.shape[0]} samples.")

## 2. Build the Similarity Graph

Connect customers whose standardised distance is below threshold.

In [ ]:
threshold = 0.8
dists = np.sqrt(((X[:,None,:] - X[None,:,:])**2).sum(axis=2))
A = (dists < threshold).astype(float)
np.fill_diagonal(A, 0)
print(f'Graph edges:    {int(A.sum()//2)}')
print(f'Average degree: {A.sum(axis=1).mean():.1f}')
print(f'Density:        {A.sum() / (len(X)*(len(X)-1)):.3f}')

## 3. Run Label Propagation

In [ ]:
lp = LabelPropagation(max_iter=100, random_state=42).fit(A)
print(f'Communities found: {lp.n_communities_}')
print(f'Converged in:      {lp.n_iter_} iterations')
print(f'Modularity:        {lp.modularity(A):.4f}')
sizes = {c: len(v) for c, v in lp.get_communities().items()}
print(f'Community sizes:   {dict(sorted(sizes.items(), key=lambda x: -x[1]))}')

## 4. Results and Visualisation

In [ ]:
pca = PCA(n_components=2).fit(X)
X_pca = pca.transform(X)

lp_labels = lp.labels_
n_comm = lp.n_communities_
cmap_lp = plt.cm.get_cmap('tab20', n_comm)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for c in range(n_comm):
    mask = lp_labels==c
    if mask.sum() > 0:
        axes[0].scatter(X_pca[mask,0], X_pca[mask,1], color=cmap_lp(c), s=30, alpha=0.75, label=f'C{c} (n={mask.sum()})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].set_title(f'Label Propagation - {n_comm} Communities (PCA view)', fontweight='bold')
axes[0].legend(fontsize=7, ncol=2)

comm_df = pd.DataFrame({'Income': X_raw[:,1], 'Spending': X_raw[:,2], 'Community': lp_labels})
for c in range(n_comm):
    sub = comm_df[comm_df['Community']==c]
    axes[1].scatter(sub['Income'], sub['Spending'], color=cmap_lp(c), s=30, alpha=0.75)
    cx, cy = sub['Income'].mean(), sub['Spending'].mean()
    axes[1].annotate(f'C{c}', (cx,cy), fontsize=8, fontweight='bold', ha='center', va='center',
                     bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
axes[1].set_xlabel('Annual Income (k$)'); axes[1].set_ylabel('Spending Score')
axes[1].set_title('Label Propagation - Income vs Spending', fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Analysis

**13 communities found, Modularity=0.729, converged in 4 iterations**

**Modularity of 0.729** is very high — values above 0.3 are considered meaningful, and above 0.7 indicates exceptionally strong community structure. This tells us the similarity graph has genuine, non-trivial groupings: the 13 communities have far more internal connections than would be expected by chance.

**13 communities from 200 customers** is a fine-grained partition compared to K-Means's 8 clusters. This happens because the threshold of 0.8 in standardised space creates a relatively sparse graph (density 0.071 = 7.1% of all possible edges exist), meaning small groups of highly similar customers form tight communities while less similar ones are separated.

**Community sizes vary dramatically** — from 39 members down to single-node communities (size 1). The three singleton communities (nodes 0, 1, 12) are the most isolated customers in feature space, similar to DBSCAN's noise points. The large communities (39, 37, 34 members) represent the dominant customer archetypes.

**Convergence in just 4 iterations** is typical for Label Propagation on well-structured graphs — the labels propagate quickly through densely connected groups and stop as soon as internal consensus is reached.

**Comparison to K-Means:** K-Means found 8 roughly equally-sized clusters by optimising inertia; Label Propagation found 13 variable-size communities by following the graph's density structure. The two approaches are complementary — K-Means is better for finding a balanced k-way partition for business use, while Label Propagation reveals the natural topology of customer similarity at a finer scale.

**Key takeaway:** The high modularity confirms that mall customers genuinely cluster by similarity — the community structure is real, not an artifact. The large number of small communities suggests the customer base is more heterogeneous than 8 clusters implies, with several niche customer types that K-Means merges into the nearest large cluster.